In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
mat_add=r"""

#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

__global__ void matrix_add(const float* A, const float* B, float* C, int rows, int cols){
    int col=blockIdx.x*blockDim.x+threadIdx.x;
    int row=blockIdx.y*blockDim.y+threadIdx.y;

    if (row<rows && col<cols){
        int idx=row*cols+col;
        C[idx]=A[idx]+B[idx];
    }
}

int main(){
    const int rows=1000, cols=1500;
    size_t bytes=rows*cols*sizeof(float);
    
    float *A, *B, *C;
    
    cudaMallocManaged(&A, bytes);
    cudaMallocManaged(&B, bytes);
    cudaMallocManaged(&C, bytes);

    dim3 block(16, 16);
    dim3 grid((cols + block.x - 1) / block.x, (rows + block.y - 1) / block.y);

    matrix_add<<<grid, block>>>(A, B, C, rows, cols);
    cudaDeviceSynchronize();

    for (int j = 0; j < 5; j++) {
        printf("C[0][%d] = %f\n", j, C[0 * cols + j]);
    }

    // last element (checks that the edge of the matrix was covered)
    printf("C[%d][%d] = %f\n", rows - 1, cols - 1, C[(rows - 1) * cols + (cols - 1)]);

    cudaFree(A);
    cudaFree(B);
    cudaFree(C);

return 0;
}

"""

with open('matrix_add.cu', 'w') as f:
    f.write(mat_add)

In [5]:
!nvcc -arch=sm_75 matrix_add.cu -o matadd

In [6]:
!./matadd

C[0][0] = 0.000000
C[0][1] = 0.000000
C[0][2] = 0.000000
C[0][3] = 0.000000
C[0][4] = 0.000000
C[999][1499] = 0.000000
